# Football Match Result Prediction with MLP — 2026 FIFA World Cup

**Course:** PSI0325 — Fundamentals of Adaptation and Machine Learning
**Student:** Jefferson Santos Monteiro (NUSP 15527878)

This notebook is **fully self-contained**: it does not import any external
project files, only standard libraries (PyTorch, pandas, NumPy, Matplotlib,
scikit-learn). It can be run top-to-bottom in Google Colab or any local
Jupyter environment with those libraries installed.

It reproduces the full pipeline: dataset construction (Elo ratings, recent
form), training an MLP and a linear baseline in PyTorch, evaluation against
the real 2026 FIFA World Cup matches (with a focus on probability
calibration, not just accuracy), and predictions for hypothetical
matchups.

## 1. Setup

In [ ]:
import io
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Loading the raw dataset

Source: the public Kaggle dataset *"International football results from
1872 to 2026"* (`results.csv`), which includes a community-maintained
extension covering the 2026 World Cup matches.

**Run the cell below and upload `results.csv` when prompted** (works in
Google Colab). If a `results.csv` is already present in the current working
directory — e.g. running locally — it is used directly instead, with no
upload prompt.

In [ ]:
import os

if os.path.exists("results.csv"):
    print("Found results.csv in the current directory, using it directly.")
    raw_df = pd.read_csv("results.csv")
else:
    try:
        from google.colab import files
        print("Please upload results.csv:")
        uploaded = files.upload()
        filename = next(iter(uploaded))
        raw_df = pd.read_csv(io.BytesIO(uploaded[filename]))
    except ImportError:
        raise FileNotFoundError(
            "results.csv not found in the current directory, and this does "
            "not appear to be a Colab environment (google.colab not "
            "available). Place results.csv next to this notebook and re-run."
        )

print(f"Loaded {len(raw_df)} matches.")
raw_df.head()

## 3. Feature engineering

Four features are used, each computed strictly from information available
**before** each match (no data leakage):

- **`elo_diff`** — Elo rating difference (home − away). Elo is computed
  *sequentially over the entire match history since 1872* (not reset in
  2000), so ratings are not artificially flattened at the start of the
  training window.
- **`home_flag`** — home advantage indicator (0 for neutral-venue matches).
- **`elo_home_interaction`** — `elo_diff × home_flag`, added to test whether
  home advantage depends on the strength gap between the two teams (a
  non-linear interaction a purely linear model cannot represent).
- **`form_diff`** — difference in points earned over each team's last 10
  matches (home − away).

**Split:** training = matches from 2000-01-01 to the eve of the 2026 World
Cup (2026-06-10); test = the 104 matches of the tournament itself
(2026-06-11 to 2026-07-19). The split is temporal, not random, so the model
never has access to future information during training.

Elo ratings and recent-form history are also kept (`team_state`) for use
later in Section 8, to predict hypothetical matchups between any two
teams.

In [ ]:
from collections import defaultdict, deque

TRAIN_START = pd.Timestamp("2000-01-01")
WORLD_CUP_START = pd.Timestamp("2026-06-11")
K_FACTOR = 20
FORM_WINDOW = 10
FEATURES = ["elo_diff", "home_flag", "elo_home_interaction", "form_diff"]


def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def result_label(home_score, away_score):
    if home_score > away_score:
        return 0  # home win
    if home_score == away_score:
        return 1  # draw
    return 2  # away win


def build_features(raw_df):
    df = raw_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    elo = defaultdict(lambda: 1500.0)
    recent_results = defaultdict(lambda: deque(maxlen=FORM_WINDOW))

    rows = []
    for _, m in df.iterrows():
        home, away = m["home_team"], m["away_team"]

        elo_diff = elo[home] - elo[away]
        home_flag = 0 if m["neutral"] else 1
        form_home = sum(recent_results[home]) if recent_results[home] else 0
        form_away = sum(recent_results[away]) if recent_results[away] else 0
        form_diff = form_home - form_away

        label = result_label(m["home_score"], m["away_score"])

        if m["date"] >= TRAIN_START:
            rows.append({
                "date": m["date"],
                "home_team": home,
                "away_team": away,
                "tournament": m["tournament"],
                "elo_diff": elo_diff,
                "home_flag": home_flag,
                "elo_home_interaction": elo_diff * home_flag,
                "form_diff": form_diff,
                "result": label,
            })

        exp_home = expected_score(elo[home], elo[away])
        score_home = 1.0 if label == 0 else (0.5 if label == 1 else 0.0)
        elo[home] += K_FACTOR * (score_home - exp_home)
        elo[away] += K_FACTOR * ((1 - score_home) - (1 - exp_home))

        pts_home = 3 if label == 0 else (1 if label == 1 else 0)
        pts_away = 3 if label == 2 else (1 if label == 1 else 0)
        recent_results[home].append(pts_home)
        recent_results[away].append(pts_away)

    full_df = pd.DataFrame(rows)
    return full_df, elo, recent_results


full_df, team_elo, team_recent_results = build_features(raw_df)

is_wc26 = (full_df["tournament"] == "FIFA World Cup") & (full_df["date"] >= WORLD_CUP_START)
test_df = full_df[is_wc26].copy()
train_df = full_df[(full_df["date"] < WORLD_CUP_START) & (~is_wc26)].copy()

print(f"Training matches: {len(train_df)} ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
print(f"Test matches (2026 World Cup): {len(test_df)}")
train_df.head()

In [ ]:
train_df["result"].value_counts(normalize=True).rename(
    index={0: "home_win", 1: "draw", 2: "away_win"}
).rename("proportion")

## 4. Model

### 4.1 MLP architecture

A single hidden layer with a small number of units (8) was chosen
deliberately. With only 4 input features and roughly 25k training examples,
a wide/deep network offers little extra representational power and mainly
increases overfitting risk — this is corroborated by the linear-vs-MLP
comparison in Section 6. Kaiming (He) initialization is used for the ReLU
hidden layer, and Xavier initialization for the output layer that feeds
into softmax. Dropout (0.2) and L2 weight decay are used for
regularization.

### 4.2 Linear baseline

To justify the use of non-linearity (as required by the course), a linear
baseline — logistic regression expressed as a single linear layer with no
hidden layer — is trained on the same features and compared against the MLP
using the same cross-entropy metric on the World Cup test set.

In [ ]:
class MatchOutcomeMLP(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8, num_classes=3, dropout=0.2):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, num_classes)
        self._init_weights()
        # No Softmax layer here on purpose — nn.CrossEntropyLoss expects
        # raw logits and applies log-softmax internally.

    def _init_weights(self):
        nn.init.kaiming_normal_(self.hidden.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.output(x)


class LinearBaseline(nn.Module):
    def __init__(self, input_dim=4, num_classes=3):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)
        # No hidden layer, no activation function: this model can only
        # learn linear decision boundaries between the 3 classes.

    def forward(self, x):
        return self.linear(x)


print(MatchOutcomeMLP())
print()
print(LinearBaseline())

## 5. Training

Both models are trained with the Adam optimizer and cross-entropy loss.
A chronological 10% slice of the training set (the most recent matches
before the split) is held out purely to monitor overfitting during
training — this is separate from, and never used to tune against, the
World Cup test set.

In [ ]:
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.1

split_idx = int(len(train_df) * (1 - VAL_FRACTION))
train_split_df, val_split_df = train_df.iloc[:split_idx], train_df.iloc[split_idx:]

X_train_raw = train_split_df[FEATURES].values.astype(np.float32)
y_train = train_split_df["result"].values.astype(np.int64)
X_val_raw = val_split_df[FEATURES].values.astype(np.float32)
y_val = val_split_df["result"].values.astype(np.int64)

feature_mean = X_train_raw.mean(axis=0)
feature_std = X_train_raw.std(axis=0)

X_train = (X_train_raw - feature_mean) / feature_std
X_val = (X_val_raw - feature_mean) / feature_std

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

X_val_t = torch.from_numpy(X_val).to(DEVICE)
y_val_t = torch.from_numpy(y_val).to(DEVICE)


def train_model(model, label):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    train_loss_history, val_loss_history = [], []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_ds)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_acc = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"[{label}] epoch {epoch:3d} | train_loss {train_loss:.4f} | "
                  f"val_loss {val_loss:.4f} | val_acc {val_acc:.3f}")

    return model, train_loss_history, val_loss_history


def plot_learning_curve(train_hist, val_hist, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(train_hist, label="train loss", color="tab:blue")
    ax.plot(val_hist, label="validation loss", color="tab:red")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cross-entropy loss")
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    plt.show()

In [ ]:
mlp_model, mlp_train_hist, mlp_val_hist = train_model(
    MatchOutcomeMLP(input_dim=len(FEATURES)), label="MLP")
plot_learning_curve(mlp_train_hist, mlp_val_hist, "Learning curve — MLP")

In [ ]:
linear_model, linear_train_hist, linear_val_hist = train_model(
    LinearBaseline(input_dim=len(FEATURES)), label="Linear")
plot_learning_curve(linear_train_hist, linear_val_hist, "Learning curve — linear baseline")

## 6. Evaluation on the 2026 World Cup

The primary metric is **cross-entropy** on the 104 real World Cup matches
(measuring probability quality, not just correctness), with accuracy
reported only as a secondary reference. The calibration curve is the key
result for this project's stated validation approach.

In [ ]:
CLASS_NAMES = ["home_win", "draw", "away_win"]

X_test_raw = test_df[FEATURES].values.astype(np.float32)
y_test = test_df["result"].values.astype(np.int64)
X_test = (X_test_raw - feature_mean) / feature_std
X_test_t = torch.from_numpy(X_test).to(DEVICE)
y_test_t = torch.from_numpy(y_test).to(DEVICE)


def evaluate_model(model, label):
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        ce_loss = nn.CrossEntropyLoss()(logits, y_test_t).item()
        preds = logits.argmax(dim=1).cpu().numpy()
        accuracy = (preds == y_test).mean()

    print(f"[{label}] World Cup 2026 test set ({len(test_df)} matches)")
    print(f"[{label}] Cross-entropy loss: {ce_loss:.4f}")
    print(f"[{label}] Accuracy (secondary metric): {accuracy:.3f}")
    return probs, preds, ce_loss, accuracy


mlp_probs, mlp_preds, mlp_ce, mlp_acc = evaluate_model(mlp_model, "MLP")

In [ ]:
def plot_calibration_curve(probs, y_true, n_bins=10):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")

    bin_edges = np.linspace(0, 1, n_bins + 1)
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_probs = probs[:, class_idx]
        observed = (y_true == class_idx).astype(float)
        bin_ids = np.digitize(class_probs, bin_edges[1:-1])
        pred_means, obs_means = [], []
        for b in range(n_bins):
            mask = bin_ids == b
            if mask.sum() == 0:
                continue
            pred_means.append(class_probs[mask].mean())
            obs_means.append(observed[mask].mean())
        ax.plot(pred_means, obs_means, marker="o", label=class_name)

    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_title("Calibration curve — 2026 World Cup")
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_calibration_curve(mlp_probs, y_test)

In [ ]:
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion matrix — 2026 World Cup")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    plt.show()


plot_confusion_matrix(y_test, mlp_preds)

In [ ]:
def plot_roc_curves(probs, y_true):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", label="random guess")
    for class_idx, class_name in enumerate(CLASS_NAMES):
        y_binary = (y_true == class_idx).astype(int)
        fpr, tpr, _ = roc_curve(y_binary, probs[:, class_idx])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{class_name} (AUC = {roc_auc:.2f})")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC curves (one-vs-rest) — 2026 World Cup")
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_roc_curves(mlp_probs, y_test)

### 6.1 Discussion

- **Calibration:** the home-win and away-win curves track the diagonal
  reasonably well, indicating the predicted probabilities are broadly
  trustworthy for those outcomes.
- **Draws are the clear limitation:** the confusion matrix shows the model
  never predicts a draw as the most likely outcome, and the one-vs-rest
  ROC curve for `draw` has AUC ≈ 0.50 — no better than chance. This mirrors
  a well-known difficulty in the football-prediction literature: draws lack
  a clear "signature" in strength-difference features like Elo, since they
  arise from more subtle factors (playing style, game context) that are not
  captured by `elo_diff`, `home_flag`, or `form_diff`. This limitation is
  precisely what the calibration-based evaluation was designed to surface —
  a plain accuracy score would have hidden it.

## 7. Linear baseline vs. MLP — does non-linearity help?

In [ ]:
linear_probs, linear_preds, linear_ce, linear_acc = evaluate_model(linear_model, "Linear")

comparison = pd.DataFrame([
    {"model": "Linear baseline", "cross_entropy": linear_ce, "accuracy": linear_acc},
    {"model": "MLP", "cross_entropy": mlp_ce, "accuracy": mlp_acc},
])
comparison

Interpret this table together with Section 6: if the MLP's cross-entropy is
at or below the linear baseline's, that supports the case for non-linearity
(helped here by the `elo_diff × home_flag` interaction term, which a linear
model cannot represent). If the two are close, that is itself a valid and
informative finding — it suggests that, with this compact 4-feature
representation, the relationship between inputs and outcome is close to
linear, and a much larger network would likely just overfit without
improving generalization to the World Cup.

## 8. Predicting hypothetical matchups

Using each team's most recent Elo rating and recent-form points (computed
in Section 3 and kept in `team_elo` / `team_recent_results`), the trained
MLP can estimate outcome probabilities for any hypothetical matchup, not
just matches already in the dataset.

In [ ]:
def predict_match(home_team, away_team, neutral=False):
    home_elo = team_elo[home_team]
    away_elo = team_elo[away_team]
    home_form = sum(team_recent_results[home_team]) if team_recent_results[home_team] else 0
    away_form = sum(team_recent_results[away_team]) if team_recent_results[away_team] else 0

    home_flag_value = 0 if neutral else 1
    elo_diff_value = home_elo - away_elo
    features = np.array([[
        elo_diff_value,
        home_flag_value,
        elo_diff_value * home_flag_value,
        home_form - away_form,
    ]], dtype=np.float32)

    X = (features - feature_mean) / feature_std
    mlp_model.eval()
    with torch.no_grad():
        probs = torch.softmax(mlp_model(torch.from_numpy(X).to(DEVICE)), dim=1)[0].cpu().numpy()
    return home_elo, away_elo, probs


def predict_and_print(home_team, away_team, neutral=False):
    home_elo, away_elo, probs = predict_match(home_team, away_team, neutral=neutral)
    print(f"{home_team} (Elo {home_elo:.0f}) vs {away_team} (Elo {away_elo:.0f})")
    for name, p in zip(CLASS_NAMES, probs.tolist()):
        print(f"  {name:10s}: {p:.1%}")


for home, away, neutral in [
    ("Brazil", "Spain", False),
    ("Brazil", "Germany", False),
    ("Argentina", "Spain", True),
]:
    predict_and_print(home, away, neutral=neutral)
    print()

## 9. Selected World Cup matches — model vs. actual results

A curated comparison table for presentation purposes: predicted
probabilities vs. actual outcomes for six notable matches from the
tournament.

In [ ]:
SELECTED_MATCHES = [
    ("Brazil", "Japan"),      # group stage
    ("Brazil", "Morocco"),    # group stage — ended in a draw, the model's known weak spot
    ("Brazil", "Norway"),     # group stage
    ("France", "Spain"),      # semifinal
    ("France", "England"),    # semifinal, 4-6 — highest-scoring match of the tournament
    ("Spain", "Argentina"),   # final
]

test_results = test_df.copy()
test_results["p_home_win"] = mlp_probs[:, 0]
test_results["p_draw"] = mlp_probs[:, 1]
test_results["p_away_win"] = mlp_probs[:, 2]
test_results["predicted"] = [CLASS_NAMES[p] for p in mlp_preds]
test_results["actual"] = [CLASS_NAMES[r] for r in test_results["result"]]
test_results["correct"] = test_results["predicted"] == test_results["actual"]

selected_rows = []
for home, away in SELECTED_MATCHES:
    match = test_results[(test_results["home_team"] == home) & (test_results["away_team"] == away)]
    if match.empty:
        print(f"WARNING: {home} vs {away} not found in test set — skipping")
        continue
    selected_rows.append(match.iloc[0])

output_cols = ["date", "home_team", "away_team", "actual", "predicted", "correct",
               "p_home_win", "p_draw", "p_away_win"]
selected_df = pd.DataFrame(selected_rows)[output_cols]
selected_df

## 10. Conclusion

This project trained an MLP to estimate match outcome probabilities for the
2026 FIFA World Cup, validated against the tournament's real results rather
than a random split. The model produces reasonably calibrated probabilities
for decisive outcomes (home/away win) but fails to discriminate draws — a
limitation consistent with prior football-prediction literature and
directly exposed by the calibration-based evaluation this project was
designed around. A linear baseline and an explicit interaction feature were
used to test, rather than assume, whether the MLP's non-linearity was
warranted for this compact feature set.